# 04.1 Project Structure

The goal of this notebook is to move you from "everything lives in one notebook" to "I can organize this into a small project".

Key concepts:

- separation of concerns
- dataset module
- model module
- training engine
- configuration management

## Learning Goals

After this notebook, you should be able to:

1. Understand why a notebook prototype should later be split into modules.
2. Distinguish the responsibilities of `dataset.py
3. Read a minimal reproducible project structure.
4. Split a toy training workflow into separate files.
5. Understand the common workflow of exploring in notebooks first, then extracting reusable code.

In [ ]:
import os
import sys
import tempfile
import textwrap
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

## A Typical Starting Point: Everything in One Notebook

This is completely normal during the learning stage.

The problems usually appear later:

- it becomes hard to locate reusable data-processing logic
- model definition and training loop are mixed together
- changing one thing can easily affect something else
- the project becomes harder to maintain

In [ ]:
def make_toy_data(n_samples=320):
    x = torch.randn(n_samples, 2)
    y = (x[:, 0] + 0.7 * x[:, 1] > 0).long()
    return x, y


class NotebookPrototypeNet(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=16, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def run_epoch_notebook(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for xb, yb in loader:
        with torch.set_grad_enabled(is_train):
            logits = model(xb)
            loss = loss_fn(logits, yb)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


x, y = make_toy_data()
train_ds = TensorDataset(x[:256], y[:256])
val_ds = TensorDataset(x[256:], y[256:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

model = NotebookPrototypeNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(1, 4):
    train_loss, train_acc = run_epoch_notebook(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch_notebook(model, val_loader, loss_fn, optimizer=None)
    print(
        f"prototype epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

The code above works, but the responsibilities are mixed together.

Next, we reorganize it into a small project.


## How to Split a Minimal Project

Start with this practical split:

- prepare data and return `DataLoader`
- define the model architecture
- training and validation logic
- hyperparameters and run configuration
- wire everything together and run

In [ ]:
project_root = Path(tempfile.mkdtemp(prefix="phase4_project_structure_"))
(project_root / "src").mkdir(parents=True, exist_ok=True)
(project_root / "configs").mkdir(parents=True, exist_ok=True)

files = {
    project_root / "src" / "__init__.py": "",
    project_root / "configs" / "__init__.py": "",
    project_root / "src" / "dataset.py": textwrap.dedent(
        """
        import torch
        from torch.utils.data import DataLoader, TensorDataset

        def make_toy_loaders(batch_size=32, seed=42):
            g = torch.Generator().manual_seed(seed)
            x = torch.randn(320, 2, generator=g)
            y = (x[:, 0] + 0.7 * x[:, 1] > 0).long()

            train_ds = TensorDataset(x[:256], y[:256])
            val_ds = TensorDataset(x[256:], y[256:])

            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
            return train_loader, val_loader
        """
    ).strip()
    + "\n",
    project_root / "src" / "model.py": textwrap.dedent(
        """
        import torch.nn as nn

        class TinyClassifier(nn.Module):
            def __init__(self, in_dim=2, hidden_dim=16, num_classes=2):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(in_dim, hidden_dim),
                    nn.ReLU(),
                    nn.Linear(hidden_dim, num_classes),
                )

            def forward(self, x):
                return self.net(x)
        """
    ).strip()
    + "\n",
    project_root / "src" / "engine.py": textwrap.dedent(
        """
        import torch

        def run_epoch(model, loader, loss_fn, optimizer=None):
            is_train = optimizer is not None
            model.train() if is_train else model.eval()
            total_loss = 0.0
            total_correct = 0
            total_items = 0

            for xb, yb in loader:
                with torch.set_grad_enabled(is_train):
                    logits = model(xb)
                    loss = loss_fn(logits, yb)

                if is_train:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                preds = logits.argmax(dim=1)
                total_loss += loss.item() * xb.size(0)
                total_correct += (preds == yb).sum().item()
                total_items += xb.size(0)

            total_items, total_correct
        """
    ).strip()
    + "\n",
    project_root / "configs" / "default_config.py": textwrap.dedent(
        """
        CONFIG = {
            "batch_size": 32,
            "hidden_dim": 16,
            "lr": 0.1,
            "epochs": 5,
            "seed": 42,
        }
        """
    ).strip()
    + "\n",
}

for path, content in files.items():
    path.write_text(content, encoding="utf-8")

for root, dirs, file_names in os.walk(project_root):
    rel_root = Path(root).relative_to(project_root)
    indent = "  " * len(rel_root.parts)
    label = "." if str(rel_root) == "." else rel_root.name
    print(f"{indent}{label}/")
    for file_name in sorted(file_names):
        print(f"{indent}  {file_name}")

print("project_root =", project_root)

This directory is still very small, but it already has the basic shape of something extensible.

When you later add experiments, switch models, or switch datasets, everything will not be piled into one notebook.


In [ ]:
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configs.default_config import CONFIG
from src.dataset import make_toy_loaders
from src.engine import run_epoch
from src.model import TinyClassifier

train_loader, val_loader = make_toy_loaders(batch_size=CONFIG["batch_size"], seed=CONFIG["seed"])
model = TinyClassifier(hidden_dim=CONFIG["hidden_dim"])
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=CONFIG["lr"])

history = []
for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    history.append((epoch, train_loss, train_acc, val_loss, val_acc))
    print(
        f"modular epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

print("final history row =", history[-1])

## When Should You Extract Modules?

A practical rule of thumb:

- the same code has been copied more than once
- the training workflow is longer than what you can comfortably understand at a glance
- you need to rerun experiments many times
- you need to explain the project structure to someone else

In [ ]:
# Exercise 1
# If you write a custom Dataset class, which file is the best place for it?
#
# A. model.py
# B. dataset.py
# C. train.py

Exercise 1 Reference Answer

- The correct answer is `B. dataset.py`

Because its responsibility is data preparation, not model definition or training orchestration.


In [ ]:
# Exercise 2
# Which file is each of the following most suitable for?
#1. nn.Module subclass / an nn.Module subclass
# a train_one_epoch function
# batch_size and learning rate

Exercise 2 Reference Answer

1. `model.py`
2. `engine.py` or `train_utils.py`
3. `config.py` or a config file

This is not the only valid split, but the responsibilities should stay stable.


## Summary

The most important takeaways from this notebook are:

1. notebook prototypes are great for exploration, but not ideal for long-term maintenance
2. config` split is very common
3. after extracting modules, the training workflow becomes clearer and more reusable
4. you do not need to over-engineer from day one, but once the project becomes repetitive and more complex, you should extract modules